# Фінальний проєкт. Інфекційні захворювання

Автор: Костромицька Ірина Вікторівна

У проєкті я працюю з набором `infectious_cases`. Це статистика інфекційних захворювань за країнами й роками. Дані лежать у файлі `infectious_cases.csv`. Базу даних PostgreSQL 16 запускаю прямо в notebook через `pgserver`.

## Підготовка

Ставлю бібліотеки, запускаю PostgreSQL і підключаюся до нього.

In [1]:
# pgserver тут не ставлю: для нового Python у Colab готового пакета може не бути
%pip install -q psycopg2-binary sqlalchemy pandas

In [2]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

PGDATA = Path("/tmp/fp_pg")


def run_command(command: str) -> str:
    """Запускає команду в bash і повертає її вивід."""
    result = subprocess.run(command, shell=True, executable="/bin/bash", capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Команда не виконалась: {command}\n{result.stderr}")
    return result.stdout


def start_pgserver() -> str:
    """Запускає PostgreSQL через pgserver і повертає рядок підключення."""
    pip_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pgserver"], capture_output=True, text=True
    )
    if pip_result.returncode == 0:
        import pgserver

        return pgserver.get_server(PGDATA, cleanup_mode="stop").get_uri()

    # pgserver є лише для Python 3.9-3.12, тому в новому Colab беру окремий Python 3.12
    venv = Path("/tmp/fp_py312")
    run_command(f"{sys.executable} -m pip install -q uv")
    run_command(f"{sys.executable} -m uv venv --python 3.12 {venv}")
    run_command(f"{sys.executable} -m uv pip install -q --python {venv}/bin/python pgserver")
    start_script = "import sys, pgserver; print(pgserver.get_server(sys.argv[1], cleanup_mode=None).get_uri())"
    output = run_command(f'{venv}/bin/python -c "{start_script}" {PGDATA}')
    return output.strip().splitlines()[-1]


pg_uri = start_pgserver()
# SQLAlchemy 2 сам би взяв драйвер psycopg 3, а ми ставимо psycopg2, тому вказую його явно
engine = create_engine(pg_uri.replace("postgresql://", "postgresql+psycopg2://", 1), future=True)

# одне підключення на весь notebook, щоб SET search_path діяв на всі наступні запити
conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")
print(conn.execute(text("SELECT version()")).scalar_one())


def run_sql(query: str):
    """Виконує SQL. Якщо запит повертає рядки, показує їх таблицею."""
    result = conn.execute(text(query))
    if result.returns_rows:
        return pd.DataFrame(result.fetchall(), columns=list(result.keys()))

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## Завдання 1. Завантаження даних

Спочатку створюю схему `pandemic` і роблю її схемою за замовчуванням, щоб не писати її назву перед кожною таблицею.

In [3]:
run_sql("""
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;
""")

Тепер створюю таблицю `infectious_cases`. Кількість захворювань беру як `NUMERIC`, бо в файлі є і цілі числа, і дроби. Порожні клітинки в CSV стануть `NULL`.

In [4]:
run_sql("""
DROP TABLE IF EXISTS infectious_cases;

CREATE TABLE infectious_cases (
    entity                TEXT NOT NULL,
    code                  TEXT,
    year                  INT  NOT NULL,
    number_yaws           NUMERIC,
    polio_cases           NUMERIC,
    cases_guinea_worm     NUMERIC,
    number_rabies         NUMERIC,
    number_malaria        NUMERIC,
    number_hiv            NUMERIC,
    number_tuberculosis   NUMERIC,
    number_smallpox       NUMERIC,
    number_cholera_cases  NUMERIC
);
""")

Дані завантажую командою `COPY ... FROM STDIN`. Після цього порівнюю кількість рядків у файлі й у таблиці.

In [5]:
COPY_SQL = """
COPY infectious_cases (
    entity, code, year, number_yaws, polio_cases, cases_guinea_worm, number_rabies,
    number_malaria, number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases
)
FROM STDIN
WITH (FORMAT csv, HEADER true)
"""

from urllib.request import urlretrieve

CSV_NAME = "infectious_cases.csv"
RAW_URL = "https://raw.githubusercontent.com/IamLoren/goit-rdb-fp/main/infectious_cases.csv"

# шукаю файл поруч з notebook, потім у батьківській папці
csv_path = next((p for p in (Path(CSV_NAME), Path("..") / CSV_NAME) if p.exists()), None)

if csv_path is None:
    try:
        urlretrieve(RAW_URL, CSV_NAME)
        csv_path = Path(CSV_NAME)
    except OSError:
        # якщо з репозиторію не вийшло, прошу завантажити файл вручну
        from google.colab import files

        csv_path = Path(next(iter(files.upload())))

with open(csv_path, "r", encoding="utf-8", newline="") as file:
    with conn.connection.cursor() as cur:
        cur.copy_expert(COPY_SQL, file)

rows_in_file = len(pd.read_csv(csv_path))
rows_in_table = int(run_sql("SELECT COUNT(*) AS n FROM infectious_cases").loc[0, "n"])
print(f"У файлі {rows_in_file} рядків, у таблиці {rows_in_table}")
assert rows_in_file == rows_in_table, "Кількість рядків у файлі й у таблиці не збігається"

Saving infectious_cases.csv to infectious_cases.csv
У файлі 10521 рядків, у таблиці 10521


Дивлюся на дані, щоб зрозуміти, як вони виглядають.

In [6]:
run_sql("""
SELECT *
FROM infectious_cases
LIMIT 10
""")

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,None,6160,0,None,None,None,None,None,None
1,Afghanistan,AFG,1981,None,5859,0,None,None,None,None,None,None
2,Afghanistan,AFG,1982,None,9730,0,None,None,None,None,None,None
3,Afghanistan,AFG,1983,None,13937,0,None,None,None,None,None,None
4,Afghanistan,AFG,1984,None,3864,0,None,None,None,None,None,None
5,Afghanistan,AFG,1985,None,13867,0,None,None,None,None,None,None
6,Afghanistan,AFG,1986,None,12901,0,None,None,None,None,None,None
7,Afghanistan,AFG,1987,None,4396,0,None,None,None,None,None,None
8,Afghanistan,AFG,1988,None,2149,0,None,None,None,None,None,None
9,Afghanistan,AFG,1989,None,385,0,None,None,None,None,None,None


In [7]:
run_sql("""
SELECT
    COUNT(*)                 AS total_rows,
    COUNT(DISTINCT entity)   AS entities,
    COUNT(DISTINCT code)     AS codes,
    MIN(year)                AS first_year,
    MAX(year)                AS last_year,
    COUNT(number_rabies)     AS rows_with_rabies
FROM infectious_cases
""")

,total_rows,entities,codes,first_year,last_year,rows_with_rabies
0,10521,245,211,1920,2023,6780


Висновок. У таблиці 10 521 рядок про 245 значень `Entity` за роки з 1920 по 2023. Багато колонок майже порожні, наприклад `Number_yaws` заповнена менш ніж у 100 рядках, а `Number_rabies` у 6 780 з 10 521. Також у `Entity` лежать не лише країни, а й світ у цілому, регіони та групи країн за доходом. Для 34 значень `Entity` немає `Code`, зокрема для регіонів.

## Завдання 2. Нормалізація до 3-ї нормальної форми

Зараз у кожному рядку повторюються `Entity` і `Code`, наприклад Afghanistan з кодом AFG стоїть у кожному році. Рядок визначається країною і роком, а `Code` залежить тільки від країни, а не від року. Через це виходить багато зайвих повторів, а якщо код треба виправити, то міняти його доведеться в багатьох рядках.

Тому ділю таблицю на дві:
- `entities` з унікальними парами `Entity` і `Code`, у кожної пари свій `id`;
- `disease_cases` з показниками за роками, де замість `Entity` і `Code` стоїть `entity_id`.

Тепер кожен показник залежить від пари `entity_id` і `year`, а назва й код зберігаються один раз. У `Code` бувають порожні значення, тому в `UNIQUE` додаю `NULLS NOT DISTINCT`, інакше такі пари могли б дублюватись.

In [8]:
run_sql("""
DROP TABLE IF EXISTS disease_cases;
DROP TABLE IF EXISTS entities;

CREATE TABLE entities (
    id      INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity  TEXT NOT NULL,
    code    TEXT,
    UNIQUE NULLS NOT DISTINCT (entity, code)
);

CREATE TABLE disease_cases (
    entity_id             INT NOT NULL REFERENCES entities (id),
    year                  INT NOT NULL,
    number_yaws           NUMERIC,
    polio_cases           NUMERIC,
    cases_guinea_worm     NUMERIC,
    number_rabies         NUMERIC,
    number_malaria        NUMERIC,
    number_hiv            NUMERIC,
    number_tuberculosis   NUMERIC,
    number_smallpox       NUMERIC,
    number_cholera_cases  NUMERIC,
    PRIMARY KEY (entity_id, year)
);
""")

Спочатку заповнюю `entities`, потім `disease_cases`: щоб знайти `id`, з'єдную стару таблицю з `entities`. Через `IS NOT DISTINCT FROM` порівнюються і порожні коди.

In [9]:
run_sql("""
TRUNCATE entities RESTART IDENTITY CASCADE;

INSERT INTO entities (entity, code)
SELECT DISTINCT entity, code
FROM infectious_cases
ORDER BY entity;

INSERT INTO disease_cases (
    entity_id, year, number_yaws, polio_cases, cases_guinea_worm, number_rabies,
    number_malaria, number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases
)
SELECT
    e.id, i.year, i.number_yaws, i.polio_cases, i.cases_guinea_worm, i.number_rabies,
    i.number_malaria, i.number_hiv, i.number_tuberculosis, i.number_smallpox, i.number_cholera_cases
FROM infectious_cases AS i
JOIN entities AS e
    ON e.entity = i.entity
   AND e.code IS NOT DISTINCT FROM i.code
""")

Перевіряю, що рядки не загубились: у `disease_cases` має бути стільки ж рядків, як в оригіналі.

In [10]:
counts = run_sql("""
SELECT
    (SELECT COUNT(*) FROM infectious_cases) AS original_rows,
    (SELECT COUNT(*) FROM entities)         AS entities_rows,
    (SELECT COUNT(*) FROM disease_cases)    AS disease_cases_rows
""")
assert counts.loc[0, "original_rows"] == counts.loc[0, "disease_cases_rows"], "Після нормалізації втратилися рядки"
counts

,original_rows,entities_rows,disease_cases_rows
0,10521,245,10521


In [11]:
run_sql("""
SELECT *
FROM entities
ORDER BY id
LIMIT 5
""")

,id,entity,code
0,1,Afghanistan,AFG
1,2,Africa,None
2,3,African Region (WHO),None
3,4,Albania,ALB
4,5,Algeria,DZA


In [12]:
run_sql("""
SELECT *
FROM disease_cases
ORDER BY entity_id, year
LIMIT 5
""")

,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,1,1960,None,None,None,None,None,None,None,None,887
1,1,1965,None,None,None,None,None,None,None,None,218
2,1,1980,None,6160,0,None,None,None,None,None,None
3,1,1981,None,5859,0,None,None,None,None,None,None
4,1,1982,None,9730,0,None,None,None,None,None,None


## Завдання 3. Аналіз `Number_rabies`

Для кожної пари `Entity` і `Code` рахую середнє, мінімальне, максимальне значення і суму `Number_rabies`. Беру тільки рядки, де значення є (`IS NOT NULL`). Без цього групи без даних мали б середнє `NULL`, а при сортуванні за спаданням `NULL` у PostgreSQL іде першим. Сортую за середнім від більшого до меншого і показую 10 рядків.

In [13]:
run_sql("""
SELECT
    e.entity,
    e.code,
    ROUND(AVG(d.number_rabies), 2) AS avg_rabies,
    MIN(d.number_rabies)           AS min_rabies,
    MAX(d.number_rabies)           AS max_rabies,
    SUM(d.number_rabies)           AS sum_rabies
FROM disease_cases AS d
JOIN entities AS e ON e.id = d.entity_id
WHERE d.number_rabies IS NOT NULL
GROUP BY e.id, e.entity, e.code
ORDER BY AVG(d.number_rabies) DESC
LIMIT 10
""")

,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,World,OWID_WRL,20192.37,14075.508,24744.658,605771.111
1,Lower Middle Income (WB),None,15193.96,10202.528,19182.795,455818.7815
2,South Asia (WB),None,11729.89,7271.2754,15361.878,351896.6626
3,South-East Asia Region (WHO),None,11424.33,6806.0127,15641.958,342729.8068
4,G20,None,10189.05,6339.0796,13164.881,305671.3801
5,India,IND,8599.17,5425.8726,11121.142,257975.1984
6,Sub-Saharan Africa (WB),None,4748.27,4356.8184,5017.977,142448.0427
7,African Region (WHO),None,4649.70,4283.073,4917.784,139491.0713
8,Low Income (WB),None,3568.11,3014.617,3873.533,107043.1635
9,East Asia & Pacific (WB),None,3427.46,2009.9395,4591.0107,102823.8340


Висновок. Найбільше середнє у `World`, далі йдуть регіони й групи країн, як-от `Lower Middle Income (WB)` і `South Asia (WB)`. Це не окремі країни, а об'єднання, тому їхні числа більші. З окремих країн у топі лише India. Якщо потрібен рейтинг саме країн, треба відкинути такі об'єднання.

## Завдання 4. Різниця в роках

Для колонки `year` будую три атрибути вбудованими функціями PostgreSQL:
- `make_date(year, 1, 1)` дає дату 1 січня цього року;
- `CURRENT_DATE` це поточна дата;
- `AGE(...)` рахує проміжок між двома датами, а `DATE_PART('year', ...)` бере з нього повні роки.

Результат залежить від дня, коли запускаю запит, тому в іншому році числа будуть іншими.

In [14]:
run_sql("""
SELECT DISTINCT
    year,
    make_date(year, 1, 1) AS first_january,
    CURRENT_DATE          AS today,
    CAST(DATE_PART('year', AGE(CURRENT_DATE, make_date(year, 1, 1))) AS INT) AS year_diff
FROM disease_cases
ORDER BY year
LIMIT 10
""")

,year,first_january,today,year_diff
0,1920,1920-01-01,2026-11-08,106
1,1921,1921-01-01,2026-11-08,105
2,1922,1922-01-01,2026-11-08,104
3,1923,1923-01-01,2026-11-08,103
4,1924,1924-01-01,2026-11-08,102
5,1925,1925-01-01,2026-11-08,101
6,1926,1926-01-01,2026-11-08,100
7,1927,1927-01-01,2026-11-08,99
8,1928,1928-01-01,2026-11-08,98
9,1929,1929-01-01,2026-11-08,97


## Завдання 5. Власні функції

Роблю дві окремі функції.

Перша, `year_diff`, бере рік і повертає різницю в роках між поточною датою та 1 січня цього року. Це той самий розрахунок, що в попередньому завданні, тільки загорнутий у функцію.

Друга, `avg_cases_per_period`, бере кількість захворювань за рік і дільник та повертає середню кількість за період. Додаю `NULLIF(divisor, 0)`, щоб при діленні на нуль вийшов `NULL`, а не помилка.

In [15]:
run_sql("""
CREATE OR REPLACE FUNCTION year_diff(input_year INT)
RETURNS INT
LANGUAGE sql
STABLE
AS $$
    SELECT CAST(DATE_PART('year', AGE(CURRENT_DATE, make_date(input_year, 1, 1))) AS INT)
$$;

CREATE OR REPLACE FUNCTION avg_cases_per_period(yearly_cases NUMERIC, divisor INT)
RETURNS NUMERIC
LANGUAGE sql
IMMUTABLE
AS $$
    SELECT yearly_cases / NULLIF(divisor, 0)
$$;
""")

Перевіряю функції на простих числах. Для `year_diff` беру 1996 з прикладу в умові, тобто різниця між сьогоднішньою датою і 1996-01-01. Для другої функції беру 120 захворювань на рік і дільники з умови: 12 (місяць), 4 (квартал) і 2 (півріччя). Має вийти 10, 30 і 60.

In [16]:
run_sql("""
SELECT
    year_diff(1996)                  AS years_since_1996,
    avg_cases_per_period(120, 12)    AS per_month,
    avg_cases_per_period(120, 4)     AS per_quarter,
    avg_cases_per_period(120, 2)     AS per_half_year
""")

,years_since_1996,per_month,per_quarter,per_half_year
0,30,10.0000000000000000,30.0000000000000000,60.0000000000000000


Тепер використовую обидві функції в одному запиті на даних. Беру тільки рядки, де `Number_rabies` є числом, і показую 10 рядків з найбільшими значеннями.

In [17]:
run_sql("""
SELECT
    e.entity,
    d.year,
    d.number_rabies,
    year_diff(d.year)                                   AS years_ago,
    ROUND(avg_cases_per_period(d.number_rabies, 12), 2) AS per_month,
    ROUND(avg_cases_per_period(d.number_rabies, 4), 2)  AS per_quarter,
    ROUND(avg_cases_per_period(d.number_rabies, 2), 2)  AS per_half_year
FROM disease_cases AS d
JOIN entities AS e ON e.id = d.entity_id
WHERE d.number_rabies IS NOT NULL
ORDER BY d.number_rabies DESC
LIMIT 10
""")

,entity,year,number_rabies,years_ago,per_month,per_quarter,per_half_year
0,World,1990,24744.658,36,2062.05,6186.16,12372.33
1,World,1991,24559.182,35,2046.60,6139.80,12279.59
2,World,1992,24337.137,34,2028.09,6084.28,12168.57
3,World,1993,24127.453,33,2010.62,6031.86,12063.73
4,World,1994,23972.55,32,1997.71,5993.14,11986.28
5,World,1995,23905.242,31,1992.10,5976.31,11952.62
6,World,1996,23831.625,30,1985.97,5957.91,11915.81
7,World,1997,23652.562,29,1971.05,5913.14,11826.28
8,World,1998,23415.79,28,1951.32,5853.95,11707.90
9,World,1999,23185.303,27,1932.11,5796.33,11592.65


Висновок. Функції працюють на реальних даних. Наприклад, для World у 1990 році було приблизно 24 745 випадків за рік, тобто близько 2 062 на місяць, 6 186 на квартал і 12 372 на півріччя. Ці числа середні, бо в таблиці є тільки річні значення, і розподілу по місяцях немає.